In [3]:
# 라이브러리 임포트
import cv2
import numpy as np
from ultralytics import YOLO
import threading
from queue import Queue
import time
from collections import defaultdict
from datetime import datetime

# MultiStreamTracker 클래스 정의
class MultiStreamTracker:
    def __init__(self, model1_path, model2_path, video_source1, video_source2):
        self.model1 = YOLO(model1_path)
        self.model2 = YOLO(model2_path)
        
        self.cap1 = cv2.VideoCapture(video_source1)
        self.cap2 = cv2.VideoCapture(video_source2)
        
        self.width1 = int(self.cap1.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height1 = int(self.cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.width2 = int(self.cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height2 = int(self.cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        self.frame_queue1 = Queue(maxsize=30)
        self.frame_queue2 = Queue(maxsize=30)
        self.result_queue1 = Queue()
        self.result_queue2 = Queue()
        
        self.tracking_memory1 = defaultdict(dict)
        self.tracking_memory2 = defaultdict(dict)
        self.feature_db1 = {}
        self.feature_db2 = {}
        
        self.is_running = False
        self.recording = False
        self.record_start_time = None
        self.video_writer1 = None
        self.video_writer2 = None
        
    def start_recording(self):
        if not self.recording:
            self.recording = True
            self.record_start_time = time.time()
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            self.video_writer1 = cv2.VideoWriter(
                f'stream1_{timestamp}.mp4',
                fourcc,
                30.0,
                (self.width1, self.height1)
            )
            self.video_writer2 = cv2.VideoWriter(
                f'stream2_{timestamp}.mp4',
                fourcc,
                30.0,
                (self.width2, self.height2)
            )
            print("Recording started...")
            
    def stop_recording(self):
        self.recording = False
        if self.video_writer1:
            self.video_writer1.release()
            self.video_writer1 = None
        if self.video_writer2:
            self.video_writer2.release()
            self.video_writer2 = None
        print("Recording stopped.")
        
    def start(self):
        self.is_running = True
        
        threading.Thread(target=self._capture_frames, args=(self.cap1, self.frame_queue1, 1)).start()
        threading.Thread(target=self._capture_frames, args=(self.cap2, self.frame_queue2, 2)).start()
        threading.Thread(target=self._process_stream, args=(1,)).start()
        threading.Thread(target=self._process_stream, args=(2,)).start()
        threading.Thread(target=self._display_and_record_results).start()
        
        self.start_recording()
        
    def stop(self):
        self.is_running = False
        self.stop_recording()
        self.cap1.release()
        self.cap2.release()
        cv2.destroyAllWindows()
        
    def _capture_frames(self, cap, queue, stream_id):
        while self.is_running:
            ret, frame = cap.read()
            if not ret:
                break
            
            if not queue.full():
                queue.put((frame, time.time()))
            else:
                try:
                    queue.get_nowait()
                    queue.put((frame, time.time()))
                except:
                    pass
                    
    def _extract_features(self, frame, box):
        x1, y1, x2, y2 = map(int, box)
        roi = frame[y1:y2, x1:x2]
        if roi.size == 0:
            return None
        return np.mean(roi, axis=(0, 1))
        
    def _process_stream(self, stream_id):
        while self.is_running:
            frame_queue = self.frame_queue1 if stream_id == 1 else self.frame_queue2
            result_queue = self.result_queue1 if stream_id == 1 else self.result_queue2
            model = self.model1 if stream_id == 1 else self.model2
            tracking_memory = self.tracking_memory1 if stream_id == 1 else self.tracking_memory2
            feature_db = self.feature_db1 if stream_id == 1 else self.feature_db2
            other_feature_db = self.feature_db2 if stream_id == 1 else self.feature_db1
            
            if not frame_queue.empty():
                frame, timestamp = frame_queue.get()
                
                results = model.track(frame, persist=True)
                
                if results and results[0].boxes.id is not None:
                    boxes = results[0].boxes
                    
                    for box, track_id, conf in zip(boxes.xyxy, boxes.id, boxes.conf):
                        track_id = int(track_id)
                        features = self._extract_features(frame, box)
                        if features is None:
                            continue
                            
                        tracking_memory[track_id] = {
                            'features': features,
                            'last_seen': timestamp,
                            'confidence': float(conf)
                        }
                        
                        if float(conf) > 0.8:
                            feature_db[track_id] = features
                            
                        best_match_id = None
                        best_match_dist = float('inf')
                        
                        for other_id, other_features in other_feature_db.items():
                            dist = np.linalg.norm(features - other_features)
                            if dist < best_match_dist and dist < 50:
                                best_match_dist = dist
                                best_match_id = other_id
                                
                        if best_match_id is not None:
                            results[0].boxes.id[track_id] = best_match_id
                            
                    result_queue.put((frame, results, timestamp))
                    
    def _display_and_record_results(self):
        while self.is_running:
            if not self.result_queue1.empty() and not self.result_queue2.empty():
                frame1, results1, ts1 = self.result_queue1.get()
                frame2, results2, ts2 = self.result_queue2.get()
                
                if results1 and results1[0].boxes.id is not None:
                    frame1 = results1[0].plot()
                if results2 and results2[0].boxes.id is not None:
                    frame2 = results2[0].plot()
                
                if self.recording:
                    self.video_writer1.write(frame1)
                    self.video_writer2.write(frame2)
                    
                    if time.time() - self.record_start_time >= 10:
                        self.stop_recording()
                
                combined_frame = np.hstack((frame1, frame2))
                cv2.imshow('Multi-stream Tracking', combined_frame)
                
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    self.stop()
                    break

# 실행 예시
model1_path = 'yolov8n.pt'  # 모델 경로 지정
model2_path = 'yolov8n.pt'
video_source1 = "rtsp://<USER>:<PASSWORD>@<CAMERA_IP>:1554/stream1"  # 비디오 파일 경로 지정
video_source2 = "rtsp://<USER>:<PASSWORD>@<CAMERA_IP>:2554/stream1"

# 트래커 실행
tracker = MultiStreamTracker(model1_path, model2_path, video_source1, video_source2)
tracker.start()

# 20초 후 자동 종료됨
# 수동 종료를 원하면:
# time.sleep(25)
# tracker.stop()

Recording started...
